In [ ]:
# -----------------------------
function 6 
# -----------------------------
!pip install scikit-optimize
# -----------------------------
# Imports
# -----------------------------
import numpy as np
from skopt import Optimizer
from skopt.learning import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# -----------------------------
# Normalize for surrogate model
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_norm = scaler_X.fit_transform(X_train)
y_train_norm = scaler_y.fit_transform(y_train.reshape(-1,1)).ravel()

# -----------------------------
# Random Forest surrogate for BO
# -----------------------------
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)

# skopt Optimizer using Random Forest as base_estimator
optimizer_bo = Optimizer(
    dimensions=[(0.0, 1.0)]*X_train.shape[1],
    base_estimator=rf,
    acq_func="EI",  # Expected Improvement
    acq_func_kwargs={"xi": 0.01},
    random_state=42
)

# -----------------------------
# Feed initial data to optimizer
# -----------------------------
for x, y in zip(X_train, y_train):
    optimizer_bo.tell(x.tolist(), y)  # NOTE: original X, not normalized

# -----------------------------
# BO loop
# -----------------------------
n_iterations = 50
for i in range(n_iterations):
    # Suggest next point
    next_x = optimizer_bo.ask()  # in [0,1]

    # Normalize for surrogate
    next_x_norm = scaler_X.transform(np.array(next_x).reshape(1,-1))

    # Predict using Random Forest surrogate (normalized optional, RF works fine unscaled too)
    y_pred = rf.fit(X_train_norm, y_train_norm).predict(next_x_norm)[0]

    # Unnormalize
    y_pred_real = scaler_y.inverse_transform([[y_pred]])[0,0]

    print(f"Iteration {i+1}: Suggested recipe = {next_x}, Predicted score = {y_pred_real:.4f}")

    # Update optimizer with predicted value
    optimizer_bo.tell(next_x, y_pred_real)

    # Add to training set for surrogate (optional)
    X_train = np.vstack([X_train, next_x])
    y_train = np.append(y_train, y_pred_real)
    X_train_norm = scaler_X.fit_transform(X_train)
    y_train_norm = scaler_y.fit_transform(y_train.reshape(-1,1)).ravel()

# -----------------------------
# Best recipe found
# -----------------------------
best_idx = np.argmax(optimizer_bo.yi)
best_input = optimizer_bo.Xi[best_idx]
best_output = optimizer_bo.yi[best_idx]

print("\nBest recipe found:")
print(f"Ingredients: {best_input}")
print("-".join(f"{v:.6f}" for v in best_input))
print(f"Score: {best_output:.4f}")

print(f"Predicted Score: {best_output:.4f}")
